In [0]:
# Gold layer: dim_companies (Kimball company dimension, SCD Type 2)
# Source: b2b_ecommerce.silver.companies - silver is latest-state-only
#         (SCD Type 1 upserts, no __START_AT/__END_AT) since the SCD
#         responsibility moved from silver to gold, so GOLD owns the
#         change history here: each run snapshots silver's current
#         state, diffs it against the last committed snapshot and
#         opens a new SCD2 version per company whenever a tracked
#         attribute actually changes.
# Mechanism: AUTO CDC FROM SNAPSHOT into a Delta-backed streaming
#         table (the auto-merge load). Silver cannot be read as a
#         change stream (an upsert target has no change data feed),
#         but it IS a complete current-state snapshot - exactly
#         what snapshot CDC wants. Ingestion-time pattern: the view
#         is read at pipeline-run time and the run's ingestion time
#         is the snapshot version (the documented collect-free
#         pattern - SDP forbids driver-side collect() in pipeline
#         source, which the earlier lambda implementation used).
#         Idempotent by construction: an identical snapshot diffs
#         to zero events, so retries and no-op runs change nothing
#         in the table.
# Grain:  one row per (company_id, __START_AT) version. Current row
#         = __END_AT IS NULL.
# Surrogate key: company_sk = crc32(company_id | updated_at of the
#         silver row that opened the version) - deterministic, stable
#         across refreshes, unique per version, assigned at insert and
#         never rewritten for unchanged companies. crc32 returns an
#         unsigned 32-bit value (0..4,294,967,295): always non-negative
#         and SHORT, max 10 digits (user-directed 2026-11-18: short,
#         easy key values instead of 19-digit hashes). Negative values
#         stay reserved per the Kimball default-member convention (-1
#         Unknown, -2 Not Applicable, -3 Error/Invalid) - no seed rows
#         are inserted for now (user-directed 2026-11-18). At this
#         scale (hundreds of companies vs 4.3B possible values)
#         collision risk is negligible.
#         company_sk is EXCLUDED from history tracking: a silver
#         re-send that only bumps updated_at re-hashes the key in
#         place instead of opening a fake version.
# Columns: company_sk + company_id + the 13 tracked business attrs.
#         updated_at / created_at are dropped from the snapshot
#         view (gold timestamp policy: CDC plumbing and the mixed
#         ART/UTC source clocks never reach gold; updated_at only
#         feeds the surrogate-key hash). Validity = __START_AT /
#         __END_AT (the AUTO CDC convention); is_current = __END_AT
#         IS NULL.
# Quality: no expectations - silver owns row-level validation, and
#         expectations are not allowed on AUTO CDC FROM SNAPSHOT
#         flows anyway; the SCD2 windows are maintained by the
#         AUTO CDC engine.
# Layout: liquid-clustered by company_sk (dims cluster by their
#         surrogate key - KPI views join on it).
from pyspark import pipelines as dp
from pyspark.sql import functions as F

SILVER_COMPANIES = "b2b_ecommerce.silver.companies"


# Silver's complete current-state snapshot, plus the surrogate key.
# A batch temporary view: the engine reads it at each pipeline run
# (ingestion-time versioning). No driver-side actions anywhere.
@dp.temporary_view(name="companies_snapshot")
def companies_snapshot():
    return (
        spark.read.table(SILVER_COMPANIES)
        # surrogate key: crc32 of the natural key + the silver row's own
        # updated_at (the change that opened this version) - unsigned,
        # non-negative, short (max 10 digits); the "|" separator keeps the
        # two concatenated values unambiguous
        .withColumn(
            "company_sk",
            F.crc32(
                F.concat_ws(
                    "|",
                    F.col("company_id").cast("string"),
                    F.col("updated_at").cast("string"),
                )
            ),
        )
        # gold timestamp policy: CDC plumbing / source clocks stay out of gold
        .drop("updated_at", "created_at")
    )


# Delta-backed streaming table: the AUTO CDC target, clustered on
# the surrogate key (dims cluster by their surrogate key).
dp.create_streaming_table(
    name="dim_companies",
    comment=(
        "Kimball company dimension, SCD Type 2: one row per (company_id, __START_AT) "
        "version, built from the b2b_ecommerce.silver.companies current-state snapshot "
        "via AUTO CDC FROM SNAPSHOT (ingestion-time pattern) keyed on company_id. A new "
        "version opens only when a tracked attribute changes (company_sk itself never "
        "opens one); identical snapshots diff to zero events, so runs are idempotent. "
        "updated_at/created_at excluded per the gold timestamp policy. Current row: "
        "__END_AT IS NULL."
    ),
    cluster_by=["company_sk"],
)


# The auto-merge load: diff each snapshot against the last committed
# one and apply inserts/updates as SCD2 versions.
dp.create_auto_cdc_from_snapshot_flow(
    target="dim_companies",
    source="companies_snapshot",  # ingestion-time snapshot versioning
    keys=["company_id"],
    stored_as_scd_type=2,
    # company_sk alone must not open versions (updated_at-only re-sends);
    # every business attribute is tracked
    track_history_except_column_list=["company_sk"],
)